# THE DOOR THAT OPENS AT 3:13 AM — Final Assembly Pipeline

Google Colab notebook that assembles the 41 Higgsfield video shots and 12 Tamil dialogue
lines produced for *THE DOOR THAT OPENS AT 3:13 AM* into one continuous 3-minute film,
per `PRODUCTION_PACKAGE.md` sections 15-21 and 27.

**What this notebook does:**
1. Downloads every generated video clip and dialogue audio file.
2. Inspects and normalizes every clip (resolution, fps, pixel format) so they cut together
   as one continuous shoot.
3. Concatenates the clips in screenplay order into a silent master video.
4. **Synthesizes** the ambience beds, door signature sound, 3:13 audio motif, and an original
   sparse drone score — Higgsfield's audio tools in this project were speech-only (no
   standalone music/SFX model was available), so these layers are built programmatically here
   rather than generated by an AI model. Treat them as a solid placeholder mix; swap in
   licensed/recorded assets before any public release.
5. Places the 12 Tamil dialogue lines at their screenplay timecodes, ducks ambience/score
   under dialogue, and enforces the two designed absolute-silence windows.
6. Masters the mix to -14 LUFS / -1 dBTP / 48kHz and muxes the final video.
7. Runs basic QC checks (duration, decode integrity, unintended-silence scan).

**Output:** `The_Door_That_Opens_at_3_13_AM_Final.mp4`

Run cells top to bottom. Everything writes under `/content/door_313am/`.


## 1. Setup

In [ ]:
import shutil
print("ffmpeg:", shutil.which("ffmpeg"))


In [ ]:
import os, json, math, subprocess, shutil
import numpy as np
import soundfile as sf
import requests
from scipy.signal import butter, lfilter, resample

ROOT = "/opt/data/repos/StoryTelling/door_313am_work"
RAW_VIDEO = f"{ROOT}/raw_video"
NORM_VIDEO = f"{ROOT}/norm_video"
RAW_AUDIO = f"{ROOT}/raw_audio"
STEMS = f"{ROOT}/stems"
OUT = f"{ROOT}/out"

for d in (RAW_VIDEO, NORM_VIDEO, RAW_AUDIO, STEMS, OUT):
    os.makedirs(d, exist_ok=True)

FPS = 24
WIDTH, HEIGHT = 1920, 1080
SR = 48000  # audio sample rate
FINAL_MP4 = f"{OUT}/The_Door_That_Opens_at_3_13_AM_Final.mp4"

def run(cmd):
    """Run a shell command, raise on failure, return combined stdout."""
    result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if result.returncode != 0:
        print(result.stdout)
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")
    return result.stdout

print("Config ready. ROOT =", ROOT)


## 2. Shot list (screenplay order)

Every clip generated via Higgsfield (Seedance 2.0), in the exact order they cut together
per `PRODUCTION_PACKAGE.md` §1-§2. `duration` is the duration actually rendered (a few
shots were clamped up from a requested 3s to the model's 4s minimum — see the shot notes).

In [ ]:
SHOTS = [
    # shot_id, duration_seconds, source_url
    ("S1-B",  6, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154348_5441a540-5be7-4ac7-82f1-d79e4f3d4082.mp4"),
    ("S1-C",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154420_7b4f5e7b-505f-4756-8943-9fc7d36deab3.mp4"),
    ("S2-A",  6, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154347_57953f57-ced4-4ea1-8476-8c19659c7c87.mp4"),
    ("S2-B",  5, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154638_3ab8b667-6ef5-4b05-9ac3-7109eb912ebc.mp4"),
    ("S2-C",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154347_e5285897-6441-4f87-9143-fb3ac9775cad.mp4"),
    ("S2-D",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154347_24debffd-5969-4798-ad0e-83104c965e9a.mp4"),
    ("S2-E",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154348_24882251-ff03-4960-9bf0-77ceb8ccad10.mp4"),
    ("S2-F",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154347_f6e70575-4547-4f04-89e6-0c154c1119e0.mp4"),
    ("S2-G",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154348_06a62d89-f139-4cb2-b052-f69d9ee5c343.mp4"),
    ("S3-A",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154419_b61f701b-c793-42f7-b2cb-4990d839d1c8.mp4"),
    ("S3-B",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154945_7e1887e5-b741-458c-9a61-1355557d66a0.mp4"),
    ("S3-C",  8, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154945_a19477fc-d95d-4d08-a821-2c550e691d27.mp4"),
    ("S4-A",  5, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154945_03dcc942-4259-4dc7-aac2-fc051d879126.mp4"),
    ("S4-B",  5, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_155651_9a3dc986-483b-41ed-b36d-c3c3e7437a79.mp4"),
    ("S4-C",  6, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_152733_de8e99eb-f1bf-4eab-b9a0-012bec0b692d.mp4"),
    ("S4-D",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154945_bdba1ef5-60bd-4d6a-a4b4-5c93b9cf7068.mp4"),
    ("S5-A",  8, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154945_00165865-f828-487e-a58a-5f887cbf64f0.mp4"),
    ("S5-B",  6, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154945_66e67206-47f0-4590-96e4-49d072a21303.mp4"),
    ("S5-C",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_155655_f5cdf4c8-0b6b-43d8-8ebd-cc8c34f33f98.mp4"),
    ("S5-D",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154945_d18b68cf-657f-4c3f-85ae-a8206638e134.mp4"),
    ("S5-E",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_154945_462adeda-1bb7-4e93-a34b-61bfd4c2aaff.mp4"),
    ("S5-F",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160156_4aa13054-d19c-44cd-9cf3-0b609e5affcc.mp4"),
    ("S6-A",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160156_5a83513a-5049-4aa1-b92e-c20bb2ae8f6b.mp4"),
    ("S6-B",  6, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160156_f75ce40d-9ce8-455c-9fc5-4b357901e566.mp4"),
    ("S6-C",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160156_361bebce-34bd-4227-974e-ac209ccd4d08.mp4"),
    ("S6-D",  9, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160156_5beb1b41-383f-43a3-9d33-da73af2f48d3.mp4"),
    ("S7-A",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160156_5a012ce6-ef72-4205-90f0-591d42111b6d.mp4"),
    ("S7-B",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160156_d33fb902-9429-45ff-91b9-4b83f661f2ad.mp4"),
    ("S7-C",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160156_573fc611-3233-4789-9b25-044806012a51.mp4"),
    ("S7-D",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160156_bba5afe3-8faf-4852-a916-413a6ffbb534.mp4"),
    ("S7-E",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160156_7ab140fb-01f2-4db7-b6dc-5aafdfa5ccbc.mp4"),
    ("S7-F",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160920_077787fb-1792-497b-b6f1-167217a527fb.mp4"),
    ("S7-G",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160920_d2cf5165-745c-4e44-ab7d-552ec7fab3dc.mp4"),
    ("S8-A",  5, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160920_9ae01dcf-91cc-4a53-ab77-572bdbd48ecf.mp4"),
    ("S8-B",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160920_5f9513e0-e030-4b37-91ee-3aeea92eabfd.mp4"),
    ("S8-C",  5, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160920_a0e95b3c-7062-45ae-b126-077078afe907.mp4"),
    ("S9-A",  5, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160920_b393d270-282b-43b3-998a-be4be78d3450.mp4"),
    ("S9-B",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160920_d18ba94b-0d7b-4ce3-aaaf-bc82e8d48bcb.mp4"),
    ("S9-C",  4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160920_41ce19ee-2307-4263-be50-844368a9b6df.mp4"),
    ("S10-A", 4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160920_b287ac96-cc3f-4f70-9f53-f73474142177.mp4"),
    ("S10-B", 4, "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_160920_3b6d07c0-3b47-4038-893a-48aa84b44dde.mp4"),
]

# Screenplay location per shot: "bedroom" or "corridor" ambience bed.
LOCATION = {
    "S1-B": "bedroom", "S1-C": "bedroom", "S2-A": "bedroom", "S2-B": "bedroom",
    "S2-C": "bedroom", "S2-D": "bedroom", "S2-E": "bedroom", "S2-F": "bedroom",
    "S2-G": "corridor", "S3-A": "bedroom", "S3-B": "bedroom", "S3-C": "bedroom",
    "S4-A": "bedroom", "S4-B": "bedroom", "S4-C": "corridor", "S4-D": "bedroom",
    "S5-A": "corridor", "S5-B": "corridor", "S5-C": "corridor", "S5-D": "corridor",
    "S5-E": "corridor", "S5-F": "corridor", "S6-A": "corridor", "S6-B": "corridor",
    "S6-C": "corridor", "S6-D": "corridor", "S7-A": "bedroom", "S7-B": "bedroom",
    "S7-C": "bedroom", "S7-D": "bedroom", "S7-E": "bedroom", "S7-F": "bedroom",
    "S7-G": "bedroom", "S8-A": "bedroom", "S8-B": "bedroom", "S8-C": "bedroom",
    "S9-A": "bedroom", "S9-B": "corridor", "S9-C": "bedroom", "S10-A": "bedroom",
    "S10-B": "bedroom",
}

INTRO_BLACK_SEC = 2.0  # pure black + near-silence before S1-B, per screenplay 00:00-00:02

# Compute actual cumulative start time of each shot from real rendered durations.
starts = {}
t = INTRO_BLACK_SEC
for shot_id, dur, _ in SHOTS:
    starts[shot_id] = t
    t += dur
TOTAL_VIDEO_SEC = t
print(f"{len(SHOTS)} shots, intro {INTRO_BLACK_SEC}s, total silent-timeline length: {TOTAL_VIDEO_SEC:.1f}s")


## 3. Dialogue list

The 12 Tamil lines generated via Higgsfield `text2speech_v2` (ElevenLabs engine), with the
voice locks: Arjun = Fraser, Maya = Helena, Narrator = Cillian. Timecodes are the authored
screenplay targets from `PRODUCTION_PACKAGE.md` §17; because a few shots were duration-clamped,
actual shot boundaries (`starts`, above) may drift a little from these — review the assembled
cut and nudge `time_sec` here if a line lands mid-cut instead of on the intended shot.

In [ ]:
DIALOGUE = [
    # id, speaker, time_sec, text, url
    ("N1",  "narrator", 9,    "சில கதவுகள்... நாம திறக்கக்கூடாது. சில கதவுகள்... தானாகவே திறக்கும்.",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172614_37d360d7-2663-46a0-b906-8799e145bcc3.mp3"),
    ("A1",  "arjun", 32,      "யாரு...?",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172615_e10ed907-88b2-4add-9570-9ebd2c12deb9.mp3"),
    ("N2",  "narrator", 43,   "மூணு நாளா... சரியா 3:13க்கு... அதே சத்தம். அதே கதவு. ஆனா... வெளியே யாருமே இல்லை.",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172755_848cf6fa-55c0-43d8-aea6-8f0c2c5f3e3b.mp3"),
    ("A2",  "arjun", 67,      "யார்... நீ?",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172756_3140f44c-e88f-4221-afe4-9d8b20bc9027.mp3"),
    ("A3",  "arjun", 85,      "உன்னை... நான் எங்கேயோ பார்த்திருக்கேன்...",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172755_9ff3973b-99ac-4e7f-85bb-a7415970b700.mp3"),
    ("M1",  "maya", 89,       "அர்ஜுன்...",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172755_2476b988-3f4c-43c1-942b-9b53248da159.mp3"),
    ("A4",  "arjun", 104,     "இல்ல... இது... உண்மை இல்ல.",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172755_044b0304-4e4d-4406-a45c-e5d8044aadf8.mp3"),
    ("M2",  "maya", 108,      "நான் உன்னை கூட்டிட்டுப் போக வரல... நீ தான்... இங்கிருந்து வரணும்.",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172614_de2fcb29-3f5d-4505-acb7-81cf5e11e1fe.mp3"),
    ("M3",  "maya", 129,      "நேரமாச்சு...",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172755_f74e333a-2120-43af-9a05-414a0807f210.mp3"),
    ("A5",  "arjun", 132,     "என்ன நேரம்?",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172755_c94a4cd2-4900-4736-a597-39f50886f71c.mp3"),
    ("M4a", "maya", 135,      "நீ இறந்து...",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172755_53c0c199-04ce-4917-ab11-d02ad6e52e0b.mp3"),
    ("M4b", "maya", 138,      "நாலு நாள் ஆச்சு.",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172755_078b76fa-e7a3-4782-baee-29c82d5b26ec.mp3"),
    ("M5",  "maya", 178,      "இப்போதாவது... வந்துட்டியே.",
     "https://d8j0ntlcm91z4.cloudfront.net/user_3FXuOlg1HqS8yvURHbzZgj2jNLj/hf_20260821_172755_bcbbf42c-230e-4256-a618-abc8714d511b.mp3"),
]

# The two designed absolute-silence windows (screenplay-authored, seconds):
#  - just before Maya's first appearance (end of S4-B into S4-C)
#  - the 1-2s beat between "நீ இறந்து..." (M4a) and "நாலு நாள் ஆச்சு." (M4b)
# The third (immediately before the M5 final whisper) is realized structurally: no score/ambience
# stem is generated for the final 2 seconds of the film at all (see cell 13 score bed).
ABSOLUTE_SILENCE_WINDOWS = [
    (starts["S4-B"] + 4.0, starts["S4-C"] + 0.5),  # ~1.5s hush right before Maya's reveal
    (137.0, 138.0),                                  # 1s gap between M4a and M4b
]
print(f"{len(DIALOGUE)} dialogue lines configured.")


## 4. Download all assets

In [ ]:
def download(url, dest):
    if os.path.exists(dest):
        return dest
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(dest, "wb") as f:
        f.write(r.content)
    return dest

video_paths = {}
for shot_id, dur, url in SHOTS:
    dest = f"{RAW_VIDEO}/{shot_id}.mp4"
    video_paths[shot_id] = download(url, dest)
    print("video:", shot_id, "->", dest)

audio_paths = {}
for line_id, speaker, t0, text, url in DIALOGUE:
    dest = f"{RAW_AUDIO}/{line_id}.mp3"
    audio_paths[line_id] = download(url, dest)
    print("audio:", line_id, "->", dest)

print("All assets downloaded.")


## 5. Inspect clips (ffprobe)

Log resolution, fps, duration, pixel format for every clip before normalizing — this is the
"detect unusual durations / resolutions" step from §27.

In [ ]:
def ffprobe_info(path):
    cmd = (
        f'ffprobe -v error -select_streams v:0 '
        f'-show_entries stream=width,height,r_frame_rate,pix_fmt '
        f'-show_entries format=duration -of json "{path}"'
    )
    out = run(cmd)
    return json.loads(out)

report = {}
for shot_id, dur, _ in SHOTS:
    info = ffprobe_info(video_paths[shot_id])
    stream = info.get("streams", [{}])[0]
    fmt = info.get("format", {})
    report[shot_id] = {
        "width": stream.get("width"),
        "height": stream.get("height"),
        "r_frame_rate": stream.get("r_frame_rate"),
        "pix_fmt": stream.get("pix_fmt"),
        "duration": float(fmt.get("duration", 0)),
        "expected_duration": dur,
    }

for shot_id, r in report.items():
    flag = "" if abs(r["duration"] - r["expected_duration"]) < 0.5 else "  <-- DURATION MISMATCH"
    print(f"{shot_id:6s} {r['width']}x{r['height']} @ {r['r_frame_rate']:>8s} "
          f"{r['pix_fmt']:8s} {r['duration']:5.2f}s{flag}")


## 6. Normalize every clip

Scale/pad to 1920x1080, force 24fps (frame duplication, no motion-smoothing interpolation),
force yuv420p, strip any native audio (native Seedance audio was disabled at generation
time; this is a safety net in case any clip carries a stray track).

In [ ]:
def normalize_clip(src, dst):
    vf = (
        f"scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=decrease,"
        f"pad={WIDTH}:{HEIGHT}:(ow-iw)/2:(oh-ih)/2,fps={FPS},format=yuv420p"
    )
    cmd = (
        f'ffmpeg -y -v error -i "{src}" -vf "{vf}" -an '
        f'-c:v libx264 -preset medium -crf 16 -pix_fmt yuv420p "{dst}"'
    )
    run(cmd)

norm_paths = {}
for shot_id, dur, _ in SHOTS:
    dst = f"{NORM_VIDEO}/{shot_id}.mp4"
    normalize_clip(video_paths[shot_id], dst)
    norm_paths[shot_id] = dst
    print("normalized:", shot_id)

print("All clips normalized to 1920x1080 @ 24fps yuv420p.")


## 7. Build the intro black card and concatenate the silent master

Screenplay 00:00-00:02: two seconds of pure black before the clock close-up (S1-B).
No optional end title card is baked in here (per the brief, that belongs in a separate
edit pass) — the film ends on the S11 black+whisper handled entirely in the audio mix.

In [ ]:
intro_path = f"{NORM_VIDEO}/_intro_black.mp4"
run(
    f'ffmpeg -y -v error -f lavfi -i color=c=black:s={WIDTH}x{HEIGHT}:r={FPS}:d={INTRO_BLACK_SEC} '
    f'-c:v libx264 -preset medium -crf 16 -pix_fmt yuv420p "{intro_path}"'
)

concat_list_path = f"{ROOT}/concat_list.txt"
with open(concat_list_path, "w") as f:
    f.write(f"file '{intro_path}'\n")
    for shot_id, dur, _ in SHOTS:
        f.write(f"file '{norm_paths[shot_id]}'\n")

silent_master = f"{ROOT}/master_silent.mp4"
run(f'ffmpeg -y -v error -f concat -safe 0 -i "{concat_list_path}" -c copy "{silent_master}"')

info = ffprobe_info(silent_master)
actual_len = float(info["format"]["duration"])
print(f"Silent master built: {silent_master}")
print(f"Expected length: {TOTAL_VIDEO_SEC:.2f}s | Actual: {actual_len:.2f}s")


## 8. Audio synthesis toolkit

Procedural sound design for the layers Higgsfield could not generate in this project
(no standalone music/SFX model was available — see the notebook intro). Kept intentionally
restrained per the film's own philosophy: silence is a character, and the negative prompt
explicitly bans "loud horror stingers" and "cliché screaming".

In [ ]:
def pink_noise(n_samples, seed=0):
    rng = np.random.default_rng(seed)
    white = rng.normal(0, 1, n_samples + 1)
    # simple 1/f approximation via cumulative-sum + high-pass
    pink = np.cumsum(white)
    pink = pink - np.mean(pink)
    pink = pink / (np.max(np.abs(pink)) + 1e-9)
    b, a = butter(2, 20 / (SR / 2), btype="high")
    pink = lfilter(b, a, pink)
    return pink[:n_samples]

def lowpass(x, cutoff_hz, order=4):
    b, a = butter(order, cutoff_hz / (SR / 2), btype="low")
    return lfilter(b, a, x)

def bandpass(x, low_hz, high_hz, order=4):
    b, a = butter(order, [low_hz / (SR / 2), high_hz / (SR / 2)], btype="band")
    return lfilter(b, a, x)

def sine(freq, dur_sec, amp=1.0, phase=0.0):
    t = np.arange(int(dur_sec * SR)) / SR
    return amp * np.sin(2 * np.pi * freq * t + phase)

def envelope(n, attack=0.05, release=0.3):
    env = np.ones(n)
    a = int(attack * SR)
    r = int(release * SR)
    if a > 0:
        env[:a] = np.linspace(0, 1, a)
    if r > 0 and r < n:
        env[-r:] *= np.linspace(1, 0, r)
    return env

def room_tone(dur_sec, fan_hz=2.2, hum_level=0.015, seed=1):
    n = int(dur_sec * SR)
    base = lowpass(pink_noise(n, seed=seed), 700) * 0.05
    t = np.arange(n) / SR
    fan_mod = 1.0 + 0.15 * np.sin(2 * np.pi * fan_hz * t)
    hum = hum_level * np.sin(2 * np.pi * 60 * t)
    return (base * fan_mod + hum) * 0.6

def corridor_tone(dur_sec, seed=2):
    n = int(dur_sec * SR)
    base = lowpass(pink_noise(n, seed=seed), 500) * 0.045
    t = np.arange(n) / SR
    # slightly more reflective / hollow than the bedroom: a subtle slow amplitude drift
    drift = 1.0 + 0.08 * np.sin(2 * np.pi * 0.05 * t)
    return base * drift * 0.6

def door_click(seed=3):
    n = int(0.06 * SR)
    burst = bandpass(pink_noise(n, seed=seed), 800, 4000) * envelope(n, 0.001, 0.05)
    return burst * 0.5

def door_creak(dur_sec=0.8, seed=4):
    n = int(dur_sec * SR)
    t = np.arange(n) / SR
    sweep_freq = 180 - 60 * (t / dur_sec)
    tone = np.sin(2 * np.pi * np.cumsum(sweep_freq) / SR)
    noise = bandpass(pink_noise(n, seed=seed), 150, 900) * 0.3
    return (tone * 0.15 + noise) * envelope(n, 0.05, 0.3)

def footstep(seed=5):
    n = int(0.12 * SR)
    thump = sine(60, n / SR, amp=0.4) * envelope(n, 0.002, 0.1)
    tap = bandpass(pink_noise(n, seed=seed), 300, 1500) * envelope(n, 0.001, 0.05) * 0.15
    return thump + tap

def motif_313():
    """Three near-inaudible low tonal pulses: low tone, lower tone, very soft resonance."""
    parts = []
    for freq, amp, dur, gap in [(55, 0.05, 0.9, 0.35), (41, 0.045, 0.9, 0.35), (30, 0.03, 1.4, 0.0)]:
        n = int(dur * SR)
        tone = sine(freq, dur, amp=amp) * envelope(n, 0.15, 0.6)
        parts.append(tone)
        if gap > 0:
            parts.append(np.zeros(int(gap * SR)))
    return np.concatenate(parts)

def maya_presence_texture(dur_sec=3.0, seed=6):
    """Extremely subtle, non-cliche: filtered noise breath + faint low pressure, no melody."""
    n = int(dur_sec * SR)
    breath = bandpass(pink_noise(n, seed=seed), 400, 1800) * envelope(n, 0.5, 1.5) * 0.03
    pressure = sine(38, dur_sec, amp=0.02) * envelope(n, 0.8, 1.5)
    return breath + pressure

def score_drone(dur_sec, level=1.0, seed=7):
    """Original sparse score bed: deep sub-bass drone + low detuned strings-like tone.
    No drums, no melody, no recognizable motif -- per the negative prompt."""
    n = int(dur_sec * SR)
    if n <= 0 or level <= 0:
        return np.zeros(max(n, 0))
    t = np.arange(n) / SR
    sub = np.sin(2 * np.pi * 36 * t) * 0.4
    string1 = np.sin(2 * np.pi * 110 * t + 0.3 * np.sin(2 * np.pi * 0.07 * t)) * 0.12
    string2 = np.sin(2 * np.pi * 113.5 * t + 0.3 * np.sin(2 * np.pi * 0.05 * t)) * 0.10
    shimmer = lowpass(pink_noise(n, seed=seed), 300) * 0.04
    drone = sub + string1 + string2 + shimmer
    fade = min(2.0, dur_sec / 4)
    drone *= envelope(n, fade, fade)
    return drone * level * 0.5

print("Synthesis toolkit ready.")


## 9. Build the ambience bed (Layer A/B, §15)

One continuous stem across the whole runtime, blended per-shot between the bedroom and
corridor tone (crossfaded at shot boundaries) so no cut ever has a hard ambience jump —
this is the primary tool the production package calls out for hiding clip joins.

In [ ]:
TOTAL_SEC = math.ceil(TOTAL_VIDEO_SEC) + 3  # pad a little past picture lock for the audio-only tail
ambience = np.zeros(int(TOTAL_SEC * SR))

CROSSFADE = 0.6  # seconds, blended at every shot boundary

segments = [("_intro", 0.0, INTRO_BLACK_SEC, "bedroom")]
for shot_id, dur, _ in SHOTS:
    segments.append((shot_id, starts[shot_id], starts[shot_id] + dur, LOCATION[shot_id]))

for i, (shot_id, t0, t1, loc) in enumerate(segments):
    seg_len = t1 - t0
    gen = room_tone if loc == "bedroom" else corridor_tone
    seg = gen(seg_len + CROSSFADE, seed=hash(shot_id) % 1000)
    n0 = int(t0 * SR)
    n1 = n0 + len(seg)
    if n1 > len(ambience):
        seg = seg[: len(ambience) - n0]
        n1 = len(ambience)
    fade_in = envelope(len(seg), attack=CROSSFADE if i > 0 else 0.0, release=0.0)
    ambience[n0:n1] += seg * fade_in

ambience = ambience / (np.max(np.abs(ambience)) + 1e-9) * 0.5
sf.write(f"{STEMS}/ambience.wav", ambience, SR)
print("Ambience bed written:", f"{STEMS}/ambience.wav", f"({len(ambience)/SR:.1f}s)")


## 10. Build the original score bed (§16 music dynamic map)

Implements the volume/intensity curve from the screenplay's music dynamic map directly —
almost silent at open, receding to near-nothing at Maya's reveal, dropping out completely
for "நீ இறந்து... நாலு நாள் ஆச்சு", and fading out completely before the final shot.

In [ ]:
# (start_sec, end_sec, level) -- level 0 = silence, 1 = fullest (still very restrained) score presence
DYNAMIC_MAP = [
    (0,   25,  0.00),
    (25,  50,  0.06),
    (50,  70,  0.03),   # tension built through silence into Maya's reveal
    (70,  95,  0.10),   # distant atmospheric texture after the reveal
    (95,  115, 0.16),   # reversed textures + subtle low string
    (115, 135, 0.00),   # music drops for "நான் உன்னை கூட்டிட்டுப்..." dialogue
    (135, 138, 0.00),   # absolute silence window (M4a -> M4b)
    (138, 140, 0.10),   # one subtle emotional low-string tone returns
    (140, 165, 0.14),   # slightly emotional, still dark
    (165, 176, 0.00),   # gradually disappears before the final reveal
    (176, TOTAL_SEC, 0.00),  # nothing under the final black + whisper
]

score = np.zeros(int(TOTAL_SEC * SR))
for t0, t1, level in DYNAMIC_MAP:
    seg = score_drone(t1 - t0, level=level, seed=int(t0) + 1)
    n0, n1 = int(t0 * SR), int(t0 * SR) + len(seg)
    n1 = min(n1, len(score))
    score[n0:n1] += seg[: n1 - n0]

sf.write(f"{STEMS}/score.wav", score, SR)
print("Score bed written:", f"{STEMS}/score.wav")


## 11. Build the SFX layer

Door signature sound (§12) at every door beat, the 3:13 audio motif (§13) at every 3:13
clock moment, footstep cues at the barefoot beats, and the Maya presence texture (§14)
under her key appearances.

In [ ]:
sfx = np.zeros(int(TOTAL_SEC * SR))

def place(buf, at_sec, gain=1.0):
    n0 = int(at_sec * SR)
    n1 = n0 + len(buf)
    if n0 < 0 or n0 >= len(sfx):
        return
    n1 = min(n1, len(sfx))
    sfx[n0:n1] += buf[: n1 - n0] * gain

# Door signature: click + creak at every door-movement beat (S1-B click, S2-C open,
# S3-A/S3-B montage, S4-B, S9-C, S10-A close).
DOOR_BEATS = ["S1-B", "S2-C", "S3-A", "S3-B", "S4-B", "S9-C", "S10-A"]
for shot_id in DOOR_BEATS:
    t0 = starts[shot_id]
    place(door_click(seed=hash(shot_id) % 1000), t0 + 0.15)
    place(door_creak(seed=hash(shot_id + "c") % 1000), t0 + 0.3, gain=0.8)

# 3:13 motif at every clock-reads-3:13 beat.
MOTIF_BEATS = ["S1-B", "S3-A", "S3-B", "S4-B"]
for shot_id in MOTIF_BEATS:
    place(motif_313(), starts[shot_id] + 0.5)

# Barefoot footstep cues.
FOOTSTEP_BEATS = ["S2-F", "S2-G", "S8-C", "S9-A"]
for shot_id in FOOTSTEP_BEATS:
    t0 = starts[shot_id]
    for k in range(3):
        place(footstep(seed=hash(shot_id) % 1000 + k), t0 + 0.4 + k * 0.55, gain=0.6)

# Maya presence texture under her key appearances.
MAYA_BEATS = ["S4-C", "S5-A", "S5-B", "S5-F", "S6-D"]
for shot_id in MAYA_BEATS:
    dur = dict((s, d) for s, d, _ in SHOTS)[shot_id]
    place(maya_presence_texture(dur_sec=dur, seed=hash(shot_id) % 1000), starts[shot_id], gain=0.7)

sfx = np.clip(sfx, -1.0, 1.0)
sf.write(f"{STEMS}/sfx.wav", sfx, SR)
print("SFX layer written:", f"{STEMS}/sfx.wav")


## 12. Build the dialogue track

Loads each Tamil line, converts to the mix sample rate, and places it at its screenplay
timecode. Also records each line's active window so the mix stage (next cell) can duck
ambience/score under it.

In [ ]:
def load_mp3_as_wav_array(path):
    wav_path = path.replace(".mp3", ".wav")
    run(f'ffmpeg -y -v error -i "{path}" -ar {SR} -ac 1 "{wav_path}"')
    data, sr = sf.read(wav_path)
    assert sr == SR
    return data

dialogue_track = np.zeros(int(TOTAL_SEC * SR))
dialogue_windows = []  # (start_sec, end_sec) per line, for ducking

for line_id, speaker, t0, text, url in DIALOGUE:
    data = load_mp3_as_wav_array(audio_paths[line_id])
    n0 = int(t0 * SR)
    n1 = n0 + len(data)
    if n1 > len(dialogue_track):
        data = data[: len(dialogue_track) - n0]
        n1 = len(dialogue_track)
    dialogue_track[n0:n1] += data
    dialogue_windows.append((t0, t0 + len(data) / SR))
    print(f"{line_id:4s} [{speaker:9s}] placed at {t0:6.1f}s, duration {len(data)/SR:.2f}s")

sf.write(f"{STEMS}/dialogue.wav", dialogue_track, SR)
print("Dialogue track written:", f"{STEMS}/dialogue.wav")


## 13. Mix

Duck ambience+score 4-8dB under every dialogue line (§21), enforce the two absolute-silence
windows (§15) by zeroing every non-essential stem across them, sum to one stereo mix.

In [ ]:
def db_to_lin(db):
    return 10 ** (db / 20)

DUCK_DB = 6.0  # within the 4-8dB range specified in the production package
DUCK_FADE = 0.25  # seconds

def build_duck_curve(n_samples, windows, duck_db, fade_sec):
    curve = np.ones(n_samples)
    fade_n = int(fade_sec * SR)
    duck_lin = db_to_lin(-duck_db)
    for (t0, t1) in windows:
        n0, n1 = int(t0 * SR), int(t1 * SR)
        n0 = max(0, n0 - fade_n)
        n1 = min(n_samples, n1 + fade_n)
        seg_len = n1 - n0
        if seg_len <= 0:
            continue
        ramp = np.ones(seg_len)
        f = min(fade_n, seg_len // 2)
        if f > 0:
            ramp[:f] = np.linspace(1.0, duck_lin, f)
            ramp[-f:] = np.linspace(duck_lin, 1.0, f)
        ramp[f: seg_len - f] = duck_lin
        curve[n0:n1] = np.minimum(curve[n0:n1], ramp)
    return curve

n_samples = int(TOTAL_SEC * SR)
duck_curve = build_duck_curve(n_samples, dialogue_windows, DUCK_DB, DUCK_FADE)

ambience_ducked = ambience[:n_samples] * duck_curve
score_ducked = score[:n_samples] * duck_curve
sfx_full = sfx[:n_samples]  # Foley/door/motif ride at full level; they're sync'd to picture, not ducked
dialogue_full = dialogue_track[:n_samples]

# Absolute silence windows: zero every stem except the required line across the window.
for (t0, t1) in ABSOLUTE_SILENCE_WINDOWS:
    n0, n1 = int(t0 * SR), int(t1 * SR)
    ambience_ducked[n0:n1] = 0.0
    score_ducked[n0:n1] = 0.0
    sfx_full[n0:n1] = 0.0
    # dialogue_full is intentionally left alone: neither window has a line inside it

mono_mix = ambience_ducked + score_ducked + sfx_full * 0.8 + dialogue_full * 1.0
mono_mix = mono_mix / (np.max(np.abs(mono_mix)) + 1e-9) * 0.9

stereo_mix = np.stack([mono_mix, mono_mix], axis=1)  # centered mono-to-stereo; widen in mastering if desired
mix_path = f"{STEMS}/mix_prelimiter.wav"
sf.write(mix_path, stereo_mix, SR)
print("Preliminary mix written:", mix_path)


## 14. Master: loudness normalize + true-peak limit

Two-pass `loudnorm` to hit the production target: -14 LUFS integrated, -1 dBTP, 48kHz stereo.

In [ ]:
mastered_wav = f"{OUT}/mastered_audio.wav"

# Pass 1: measure
measure_out = run(
    f'ffmpeg -y -v error -i "{mix_path}" -af '
    f'loudnorm=I=-14:TP=-1:LRA=11:print_format=json -f null -'
)
# ffmpeg prints the JSON block after other stderr; extract it.
json_start = measure_out.rfind("{")
measured = json.loads(measure_out[json_start:])
print("Measured:", measured)

# Pass 2: apply using measured values for a linear, single-pass-accurate normalize
run(
    f'ffmpeg -y -v error -i "{mix_path}" -af '
    f'loudnorm=I=-14:TP=-1:LRA=11:'
    f'measured_I={measured["input_i"]}:measured_TP={measured["input_tp"]}:'
    f'measured_LRA={measured["input_lra"]}:measured_thresh={measured["input_thresh"]}:'
    f'offset={measured["target_offset"]}:linear=true:print_format=summary '
    f'-ar {SR} -ac 2 "{mastered_wav}"'
)
print("Mastered audio written:", mastered_wav)


## 15. Mux final video

In [ ]:
run(
    f'ffmpeg -y -v error -i "{silent_master}" -i "{mastered_wav}" '
    f'-map 0:v:0 -map 1:a:0 -c:v copy -c:a aac -b:a 256k -shortest "{FINAL_MP4}"'
)
print("Final film muxed:", FINAL_MP4)


## 16. QC checks (§27 steps 19-22)

Duration sanity, decode-integrity pass, and an unintended-silence scan (flags any silence
run longer than 1.5s that falls outside the two designed absolute-silence windows and the
final black-screen tail).

In [ ]:
# 1) Duration check
info = ffprobe_info(FINAL_MP4)
final_dur = float(info["format"]["duration"])
print(f"Final duration: {final_dur:.2f}s (video timeline target ~{TOTAL_VIDEO_SEC:.1f}s + audio tail)")

# 2) Decode integrity: fully decode video+audio, fail loudly on any broken/corrupt frame
decode_check = subprocess.run(
    f'ffmpeg -v error -i "{FINAL_MP4}" -f null -',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
if decode_check.stdout.strip():
    print("DECODE WARNINGS/ERRORS:\n", decode_check.stdout)
else:
    print("Decode check: no errors, no broken/stuck frames detected.")

# 3) Unintended-silence scan
silence_out = run(
    f'ffmpeg -v error -i "{FINAL_MP4}" -af silencedetect=noise=-40dB:d=1.5 -f null - 2>&1 || true'
)
expected_windows = ABSOLUTE_SILENCE_WINDOWS + [(TOTAL_SEC - 3, TOTAL_SEC)]
print("Detected silence windows (review against the two designed silences + final tail):")
for line in silence_out.splitlines():
    if "silence_start" in line or "silence_end" in line:
        print(" ", line.strip())

print("\nQC pass complete. Manually review the final MP4 for: character consistency across")
print("cuts, clip-boundary matching, and whether the synthesized score/ambience/SFX read as")
print("intended -- these are a solid placeholder mix, not a substitute for a final sound pass.")


## 17. Notes / known limitations

- **Score, ambience, and Foley are synthesized placeholders**, not AI-generated or recorded
  audio — Higgsfield's audio tools available in this project were speech-only. They follow
  the production package's design intent (restrained, no cliché stingers, silence-forward)
  but a professional sound pass (or a proper music/SFX generation tool) should replace them
  before any public release, per the monetization/copyright checklist (§26).
- **Dialogue timecodes are the authored screenplay targets**, not measured against final
  shot boundaries frame-by-frame. A few shots were duration-clamped by the generation API
  (3s requests became 4s), so watch for lines landing a beat early/late against the picture
  and adjust `DIALOGUE` timecodes in cell 3 if needed, then re-run from cell 12 onward.
  Regenerating from cell 12 does not require re-downloading or re-normalizing video.
- **Frozen-frame trimming** (§27 step 8) was not needed here since every clip's rendered
  duration matched its requested duration (see the ffprobe report in cell 5); the pipeline
  did not have to trim trailing static frames.
- **No optional end title card** is baked into this cut — add it as a separate short clip
  appended after `FINAL_MP4` if wanted, per the brief's "+2-3s, edit only" instruction.
